# Small study — neural vs classical enhancement and AI-image detector false positives

Reduced version for the 25 Oct deadline: **300 test images, 14 variants, 3 detectors** — about 2 GPU-hours.
Everything you already made (photos, fakes, pilot outputs) on Drive is reused.

| Step | What | Approx. time (T4 GPU) |
|---|---|---|
| 0 | Setup | 3 min |
| 1 | Check photos and fakes are on Drive | 1 min |
| 2 | Enhancement variants | ~1–1.5 h |
| 3 | Fidelity (PSNR, SSIM, LPIPS, DISTS) | ~10 min |
| 4 | Detector scores | ~30 min |
| 5 | Analysis, tables, figures, download | 2 min |

Runtime → Change runtime type → **T4 GPU**. If Colab disconnects: run Step 0 again, then the step you were on — it resumes.

## Step 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['STUDY_ROOT'] = '/content/drive/MyDrive/aigi_study'
os.environ['STUDY_N_TEST'] = '300'      # small study: first 300 test images
CODE = '/content/aigi_code'
os.makedirs(CODE, exist_ok=True)

In [ ]:
!pip -q install open_clip_torch lpips piq spandrel spandrel_extra_arches bm3d statsmodels scikit-image opencv-python-headless

In [ ]:
%%writefile /content/aigi_code/common.py
"""Shared paths, image I/O and reproducibility helpers for the enhancement study.

Every image (real, fake, or processed) passes through the same crop -> PNG
pipeline so that no format or size difference can explain a detector change.
"""
import hashlib
import json
import os
import random
from pathlib import Path

import numpy as np
from PIL import Image

# Root folder. On Colab point this at Google Drive so work survives disconnects.
ROOT = Path(os.environ.get("STUDY_ROOT", "/content/drive/MyDrive/aigi_study"))

DIRS = {
    "clean": ROOT / "clean",          # 512x512 PNG crops of real photos
    "variants": ROOT / "variants",    # variants/<variant_name>/<image_id>.png
    "fakes": ROOT / "fakes",          # 512x512 PNG crops of AI-generated images
    "scores": ROOT / "scores",        # one CSV of detector scores per detector
    "results": ROOT / "results",      # tables + figures for the paper
    "weights": ROOT / "weights",      # downloaded model checkpoints
}

CROP = 512            # final image side in pixels
RESIZE_SHORT = 1024   # short side before the centre crop
SEED = 2026
N_CALIB = 300         # clean reals used only to set detector thresholds


def ensure_dirs():
    for d in DIRS.values():
        d.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def image_seed(image_id, tag=""):
    """Stable per-image seed, so noise is identical across reruns."""
    h = hashlib.sha256(f"{image_id}|{tag}|{SEED}".encode()).hexdigest()
    return int(h[:8], 16)


def load_rgb(path):
    """Read an image as an HxWx3 uint8 RGB array."""
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"), dtype=np.uint8).copy()


def save_png(arr, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.ascontiguousarray(arr, dtype=np.uint8)).save(path, format="PNG")


def standard_crop(img):
    """Resize the short side to RESIZE_SHORT (antialiased Lanczos), centre-crop CROP."""
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    img = img.convert("RGB")
    w, h = img.size
    s = RESIZE_SHORT / min(w, h)
    if s < 1:  # only downsample; never upsample a small source
        img = img.resize((round(w * s), round(h * s)), Image.LANCZOS)
    w, h = img.size
    if min(w, h) < CROP:
        raise ValueError(f"image too small for a {CROP}px crop: {w}x{h}")
    left, top = (w - CROP) // 2, (h - CROP) // 2
    return np.asarray(img.crop((left, top, left + CROP, top + CROP)), dtype=np.uint8)


def list_clean_ids():
    return sorted(p.stem for p in DIRS["clean"].glob("*.png"))


def get_splits(n_calib=N_CALIB):
    """Fixed calibration/test split, saved once so every stage uses the same one."""
    path = ROOT / "splits.json"
    if path.exists():
        return _limit_test(json.loads(path.read_text()))
    ids = list_clean_ids()
    if len(ids) <= n_calib:
        raise ValueError(f"need more than {n_calib} clean images, found {len(ids)}")
    rng = random.Random(SEED)
    shuffled = ids[:]
    rng.shuffle(shuffled)
    splits = {"calib": sorted(shuffled[:n_calib]), "test": sorted(shuffled[n_calib:])}
    path.write_text(json.dumps(splits, indent=1))
    return _limit_test(splits)


def _limit_test(splits):
    """Smaller study: use only the first N_TEST test images (env STUDY_N_TEST).
    The saved split is unchanged, so the subset is deterministic and nested."""
    n = int(os.environ.get("STUDY_N_TEST", "0"))
    if n:
        splits = {"calib": splits["calib"], "test": splits["test"][:n]}
    return splits


def variant_path(variant, image_id):
    if variant == "clean":
        return DIRS["clean"] / f"{image_id}.png"
    if variant.startswith("fake_"):
        return DIRS["fakes"] / variant[5:] / f"{image_id}.png"
    return DIRS["variants"] / variant / f"{image_id}.png"


def parse_variant(name):
    """'up_realesrgan_4' -> dict(family='up', method='realesrgan', strength=4, kind='neural')."""
    from enhance import METHOD_KIND  # local import avoids a cycle
    if name == "clean":
        return {"family": "clean", "method": "clean", "strength": 0, "kind": "baseline"}
    if name.startswith("fake_"):
        return {"family": "fake", "method": name[5:], "strength": 0, "kind": "fake"}
    family, method, strength = name.split("_")
    return {"family": family, "method": method, "strength": int(strength),
            "kind": METHOD_KIND[(family, method)]}

In [ ]:
%%writefile /content/aigi_code/data.py
"""Build the clean real-photo set and the (sanity-check) fake set.

Real photos: RAISE-1k (uncompressed camera originals) is the recommended source.
Either point `prepare_from_folder` at images you already have, or give
`download_raise` the RAISE CSV of download links. DIV2K is a fallback.
"""
import csv
import random
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import requests

from common import DIRS, SEED, ensure_dirs, save_png, standard_crop

IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}


def _read_any(path):
    """Read 8- or 16-bit images (RAISE TIFFs can be 16-bit) as uint8 RGB."""
    import cv2
    arr = cv2.imread(str(path), cv2.IMREAD_COLOR | cv2.IMREAD_ANYDEPTH)
    if arr is None:
        from PIL import Image
        with Image.open(path) as im:
            return np.asarray(im.convert("RGB"), dtype=np.uint8)
    if arr.dtype == np.uint16:
        arr = (arr / 257.0).round().astype(np.uint8)
    return arr[:, :, ::-1].copy()  # BGR -> RGB


def _safe_id(prefix, stem):
    keep = "".join(c if c.isalnum() else "-" for c in stem)
    return f"{prefix}{keep}"


def prepare_from_folder(src_dir, max_n=1000, prefix="r", out_dir=None):
    """Crop every image in src_dir (recursively) to the standard 512 px PNG."""
    ensure_dirs()
    out_dir = Path(out_dir or DIRS["clean"])
    files = sorted(p for p in Path(src_dir).rglob("*") if p.suffix.lower() in IMG_EXT)
    done = 0
    for p in files:
        if done >= max_n:
            break
        out = out_dir / f"{_safe_id(prefix, p.stem)}.png"
        if out.exists():
            done += 1
            continue
        try:
            save_png(standard_crop(_read_any(p)), out)
            done += 1
        except Exception as e:  # skip unreadable or too-small files
            print("skip", p.name, e)
    print(f"{done} images ready in {out_dir}")
    return done


def download_raise(csv_path, max_n=1000, url_column=None):
    """Stream RAISE TIFFs one at a time: download -> crop -> save -> delete.

    The RAISE download page provides a CSV with one row per image and a column
    of TIFF links. Each 30 MB TIFF is deleted right after cropping, so disk use
    stays small.
    """
    ensure_dirs()
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise ValueError("empty CSV")
    if url_column is None:
        cands = [c for c in rows[0] if "tif" in c.lower()]
        if not cands:
            raise ValueError(f"no TIFF column found; columns are {list(rows[0])}. Pass url_column=...")
        url_column = cands[0]
    tmp = Path(tempfile.mkdtemp())
    done = 0
    for i, row in enumerate(rows):
        if done >= max_n:
            break
        url = row[url_column].strip()
        name = (row.get("File") or Path(url).stem or f"img{i:04d}").strip()
        out = DIRS["clean"] / f"{_safe_id('r', Path(name).stem)}.png"
        if out.exists():
            done += 1
            continue
        try:
            local = tmp / "img.tif"
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(local, "wb") as fh:
                    shutil.copyfileobj(r.raw, fh)
            save_png(standard_crop(_read_any(local)), out)
            done += 1
            if done % 50 == 0:
                print(done, "done")
        except Exception as e:
            print("skip", url, e)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"{done} RAISE images ready")
    return done


DIV2K_URLS = [
    "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip",   # 100 images
    "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip",   # 800 images
]


def download_div2k(max_n=900):
    """Fallback real set if the RAISE server is unavailable (about 900 photos)."""
    tmp = Path(tempfile.mkdtemp())
    for url in DIV2K_URLS:
        z = tmp / Path(url).name
        print("downloading", url)
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(z, "wb") as fh:
                shutil.copyfileobj(r.raw, fh)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(tmp / "x")
        z.unlink()
    n = prepare_from_folder(tmp / "x", max_n=max_n, prefix="d")
    shutil.rmtree(tmp, ignore_errors=True)
    return n


def prepare_fakes(src_dir, per_generator=100):
    """Synthbuster: one sub-folder per generator. Sample and crop like the reals.

    Saved as fakes/<generator>/<id>.png. Only used to show each detector works
    (TPR at the threshold), never to set thresholds.
    """
    rng = random.Random(SEED)
    src_dir = Path(src_dir)
    gens = sorted(d for d in src_dir.iterdir() if d.is_dir())
    for g in gens:
        files = sorted(p for p in g.rglob("*") if p.suffix.lower() in IMG_EXT)
        rng.shuffle(files)
        gname = "".join(c for c in g.name.lower() if c.isalnum())
        n = 0
        for p in files:
            if n >= per_generator:
                break
            out = DIRS["fakes"] / gname / f"{_safe_id('f', p.stem)}.png"
            if not out.exists():
                try:
                    save_png(standard_crop(_read_any(p)), out)
                except Exception as e:
                    print("skip", p.name, e)
                    continue
            n += 1
        print(f"{gname}: {n} fakes")


def fake_generators():
    return sorted(d.name for d in DIRS["fakes"].iterdir() if d.is_dir()) if DIRS["fakes"].exists() else []

In [ ]:
%%writefile /content/aigi_code/enhance.py
"""Degrade -> restore operators: classical vs neural, at graded strengths.

Every output has the clean crop as its reference, so fidelity (PSNR/SSIM/LPIPS)
can be measured and neural vs classical compared at matched fidelity.

Variant names are '<family>_<method>_<strength>', e.g. 'up_realesrgan_4'
(upscaling, factor 4) or 'dn_swinir_25' (denoising, sigma 25 on a 0-255 scale).
Runs are resumable: an output that already exists on disk is skipped.
"""
import json
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter

from common import CROP, DIRS, ROOT, image_seed, load_rgb, save_png, variant_path

# ---------------------------------------------------------------- registry
UP_SCALES = [2, 4]
DN_SIGMAS = [15, 25, 50]

METHOD_KIND = {
    ("up", "bicubic"): "classical",
    ("up", "lanczosusm"): "classical",
    ("up", "realesrgan"): "neural",
    ("up", "swinir"): "neural",
    ("up", "sdx4"): "neural",
    ("dn", "noisy"): "baseline",     # degraded input, no restoration
    ("dn", "nlm"): "classical",
    ("dn", "bm3d"): "classical",
    ("dn", "swinir"): "neural",
    ("dn", "restormer"): "neural",
}

GH = "https://github.com"
WEIGHTS = {
    ("realesrgan", 2): f"{GH}/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    ("realesrgan", 4): f"{GH}/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    ("swinir_sr", 2): f"{GH}/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x2_GAN.pth",
    ("swinir_sr", 4): f"{GH}/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth",
    ("swinir_dn", 15): f"{GH}/JingyunLiang/SwinIR/releases/download/v0.0/005_colorDN_DFWB_s128w8_SwinIR-M_noise15.pth",
    ("swinir_dn", 25): f"{GH}/JingyunLiang/SwinIR/releases/download/v0.0/005_colorDN_DFWB_s128w8_SwinIR-M_noise25.pth",
    ("swinir_dn", 50): f"{GH}/JingyunLiang/SwinIR/releases/download/v0.0/005_colorDN_DFWB_s128w8_SwinIR-M_noise50.pth",
    ("restormer_dn", 0): f"{GH}/swz30/Restormer/releases/download/v1.0/gaussian_color_denoising_blind.pth",
}


BM3D_VARIANTS = ["dn_bm3d_25"]

# Smaller study (about 2 GPU-hours on 300 test images): two upscaling factors and two
# noise levels, two classical and two neural restorers per family.
SMALL_VARIANTS = [
    "up_bicubic_2", "up_lanczosusm_2", "up_realesrgan_2", "up_swinir_2",
    "up_bicubic_4", "up_lanczosusm_4", "up_realesrgan_4", "up_swinir_4",
    "dn_noisy_15", "dn_nlm_15", "dn_swinir_15",
    "dn_noisy_50", "dn_nlm_50", "dn_swinir_50",
]


def all_variants(include_slow=True):
    v = []
    for s in UP_SCALES:
        v += [f"up_bicubic_{s}", f"up_lanczosusm_{s}", f"up_realesrgan_{s}", f"up_swinir_{s}"]
    if include_slow:
        v.append("up_sdx4_4")
    for sg in DN_SIGMAS:
        v += [f"dn_noisy_{sg}", f"dn_nlm_{sg}", f"dn_swinir_{sg}"]
        if include_slow:
            v.append(f"dn_restormer_{sg}")
    # BM3D (~36 s/image on Colab CPU) is run separately on a subset at sigma 25: see BM3D_VARIANTS
    return v


# ------------------------------------------------------------- degradations
def downscale(img, s):
    """Antialiased bicubic downscale (Pillow scales the filter support on reduction)."""
    return np.asarray(Image.fromarray(img).resize((CROP // s, CROP // s), Image.BICUBIC))


def add_noise(img, sigma, image_id):
    """Additive Gaussian noise, same draw for an image on every run, quantised to uint8."""
    rng = np.random.default_rng(image_seed(image_id, f"noise{sigma}"))
    noisy = img.astype(np.float32) + rng.normal(0.0, sigma, img.shape).astype(np.float32)
    return np.clip(noisy.round(), 0, 255).astype(np.uint8)


# ------------------------------------------------------- classical restorers
def up_bicubic(lr, s):
    return np.asarray(Image.fromarray(lr).resize((CROP, CROP), Image.BICUBIC))


def up_lanczosusm(lr, s):
    """Lanczos upscale + unsharp mask: a strong classical 'enhance' baseline."""
    up = Image.fromarray(lr).resize((CROP, CROP), Image.LANCZOS)
    return np.asarray(up.filter(ImageFilter.UnsharpMask(radius=1.5, percent=80, threshold=2)))


def _nlm(noisy, h):
    import cv2
    bgr = noisy[:, :, ::-1]
    out = cv2.fastNlMeansDenoisingColored(np.ascontiguousarray(bgr), None, h, h, 7, 21)
    return out[:, :, ::-1].copy()


def tune_nlm(calib_ids, n=10):
    """Pick the NLM strength h per sigma that maximises PSNR on calibration images.

    Makes the classical baseline as strong as it can be, so a neural-vs-classical
    gap cannot be blamed on a weak baseline. Cached in nlm_h_v2.json.
    (v1 used a grid whose optimum sat at its lower edge; outputs made with it are removed.)
    """
    from skimage.metrics import peak_signal_noise_ratio as psnr
    path = ROOT / "nlm_h_v2.json"
    old = ROOT / "nlm_h.json"
    if old.exists():  # outputs from the v1 grid used a sub-optimal h: delete and redo
        import shutil
        for d in DIRS["variants"].glob("dn_nlm_*"):
            shutil.rmtree(d)
        old.unlink()
        print("Removed NLM outputs made with the old grid; they will be regenerated.")
    if path.exists():
        return {int(k): v for k, v in json.loads(path.read_text()).items()}
    best = {}
    for sg in DN_SIGMAS:
        scores = {}
        for mult in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9, 1.1):
            h = sg * mult
            vals = []
            for iid in calib_ids[:n]:
                clean = load_rgb(variant_path("clean", iid))
                vals.append(psnr(clean, _nlm(add_noise(clean, sg, iid), h), data_range=255))
            scores[h] = float(np.mean(vals))
        best[sg] = max(scores, key=scores.get)
        print(f"sigma {sg}: best h = {best[sg]:.1f} (PSNR {scores[best[sg]]:.2f} dB)")
    path.write_text(json.dumps(best))
    return best


def _bm3d(noisy, sigma):
    import bm3d
    out = bm3d.bm3d_rgb(noisy.astype(np.float32) / 255.0, sigma_psd=sigma / 255.0)
    return np.clip(out * 255.0 + 0.5, 0, 255).astype(np.uint8)


# ---------------------------------------------------------- neural restorers
def _device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def _weights(key):
    import torch
    url = WEIGHTS[key]
    dst = DIRS["weights"] / Path(url).name
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        torch.hub.download_url_to_file(url, str(dst))
    return dst


_EXTRA_ARCHES = False


def _install_extra_arches():
    """Register spandrel's extra architectures (Restormer) exactly once.
    Calling install() a second time raises DuplicateArchitectureError."""
    global _EXTRA_ARCHES
    if _EXTRA_ARCHES:
        return
    try:
        import spandrel_extra_arches
        spandrel_extra_arches.install()
    except ImportError:
        pass
    except Exception as e:  # already registered earlier in this session
        if "Duplicate" not in type(e).__name__:
            raise
    _EXTRA_ARCHES = True


class SpandrelModel:
    """Loads Real-ESRGAN / SwinIR / Restormer checkpoints through spandrel."""

    def __init__(self, key, pad_multiple=16):
        import torch
        from spandrel import ModelLoader
        _install_extra_arches()
        self.device = _device()
        self.desc = ModelLoader(device=self.device).load_from_file(str(_weights(key)))
        self.desc.eval()
        self.scale = int(self.desc.scale)
        self.pad = pad_multiple
        self.torch = torch

    def __call__(self, img):
        torch = self.torch
        h, w = img.shape[:2]
        x = torch.from_numpy(np.array(img, copy=True)).permute(2, 0, 1).float().div(255.0)[None].to(self.device)
        ph, pw = (-h) % self.pad, (-w) % self.pad
        if ph or pw:
            x = torch.nn.functional.pad(x, (0, pw, 0, ph), mode="reflect")
        with torch.no_grad():
            y = self.desc(x)
        y = y[:, :, : h * self.scale, : w * self.scale]
        y = y.clamp(0, 1).mul(255.0).round().byte()[0].permute(1, 2, 0).cpu().numpy()
        return y


class SDUpscaler:
    """Stable Diffusion x4 upscaler (diffusion-based super-resolution), 128 -> 512."""

    def __init__(self, steps=25, noise_level=20):
        import torch
        from diffusers import StableDiffusionUpscalePipeline
        dev = _device()
        from diffusers.utils import logging as dlog
        dlog.set_verbosity_error()
        dtype = torch.float16 if dev == "cuda" else torch.float32
        name = "stabilityai/stable-diffusion-x4-upscaler"
        try:
            pipe = StableDiffusionUpscalePipeline.from_pretrained(name, dtype=dtype)
        except TypeError:  # older diffusers
            pipe = StableDiffusionUpscalePipeline.from_pretrained(name, torch_dtype=dtype)
        self.pipe = pipe.to(dev)
        self.pipe.set_progress_bar_config(disable=True)
        self.steps, self.noise_level, self.dev, self.torch = steps, noise_level, dev, torch

    def __call__(self, lr, image_id):
        g = self.torch.Generator(self.dev).manual_seed(image_seed(image_id, "sdx4"))
        out = self.pipe(prompt="", image=Image.fromarray(lr), num_inference_steps=self.steps,
                        noise_level=self.noise_level, generator=g).images[0]
        out = out.convert("RGB")
        if out.size != (CROP, CROP):
            out = out.resize((CROP, CROP), Image.BICUBIC)
        return np.asarray(out)


# ------------------------------------------------------------------ runner
def _make_op(variant, nlm_h=None):
    """Return f(clean_img, image_id) -> output image for one variant."""
    family, method, strength = variant.split("_")
    strength = int(strength)
    if family == "up":
        s = strength
        if method == "bicubic":
            return lambda img, iid: up_bicubic(downscale(img, s), s)
        if method == "lanczosusm":
            return lambda img, iid: up_lanczosusm(downscale(img, s), s)
        if method == "realesrgan":
            m = SpandrelModel(("realesrgan", s))
            return lambda img, iid: m(downscale(img, s))
        if method == "swinir":
            m = SpandrelModel(("swinir_sr", s))
            return lambda img, iid: m(downscale(img, s))
        if method == "sdx4":
            assert s == 4, "the SD upscaler is x4 only"
            m = SDUpscaler()
            return lambda img, iid: m(downscale(img, 4), iid)
    if family == "dn":
        sg = strength
        if method == "noisy":
            return lambda img, iid: add_noise(img, sg, iid)
        if method == "nlm":
            h = (nlm_h or {}).get(sg, 0.9 * sg)
            return lambda img, iid: _nlm(add_noise(img, sg, iid), h)
        if method == "bm3d":
            return lambda img, iid: _bm3d(add_noise(img, sg, iid), sg)
        if method == "swinir":
            m = SpandrelModel(("swinir_dn", sg))
            return lambda img, iid: m(add_noise(img, sg, iid))
        if method == "restormer":
            m = SpandrelModel(("restormer_dn", 0), pad_multiple=8)
            return lambda img, iid: m(add_noise(img, sg, iid))
    raise ValueError(f"unknown variant {variant}")


def run_variant(variant, image_ids, nlm_h=None):
    todo = [i for i in image_ids if not variant_path(variant, i).exists()]
    if not todo:
        print(f"{variant}: already complete")
        return
    op = _make_op(variant, nlm_h)
    t0 = time.time()
    for k, iid in enumerate(todo, 1):
        out = op(load_rgb(variant_path("clean", iid)), iid)
        assert out.shape == (CROP, CROP, 3), f"{variant} produced {out.shape}"
        save_png(out, variant_path(variant, iid))
        if k == 5:
            per = (time.time() - t0) / 5
            print(f"{variant}: {per:.2f} s/image, about {per * len(todo) / 60:.0f} min for {len(todo)}")
    print(f"{variant}: done ({len(todo)} new) in {(time.time() - t0) / 60:.1f} min")
    try:
        import torch
        del op
        torch.cuda.empty_cache()
    except Exception:
        pass


def run_variants(variants, image_ids, nlm_h=None):
    failed = {}
    for v in variants:
        try:
            run_variant(v, image_ids, nlm_h)
        except Exception as e:  # keep going; report at the end
            failed[v] = repr(e)
            print(f"!! {v} failed: {e!r}")
    if failed:
        print("\nFailed variants (fix and rerun; finished images are kept):")
        for v, e in failed.items():
            print(" ", v, e)
    return failed

In [ ]:
%%writefile /content/aigi_code/fidelity.py
"""Fidelity of every processed image to its clean original.

PSNR and SSIM (pixel fidelity), LPIPS-AlexNet and DISTS (perceptual distance).
Output: results/fidelity.csv with one row per (image_id, variant). Resumable.
"""
import numpy as np
import pandas as pd

from common import DIRS, load_rgb, variant_path


def compute_fidelity(variants, image_ids, batch=16):
    import lpips
    import piq
    import torch
    from skimage.metrics import peak_signal_noise_ratio, structural_similarity

    out_csv = DIRS["results"] / "fidelity.csv"
    old = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame(
        columns=["image_id", "variant", "psnr", "ssim", "lpips", "dists"])
    have = set(zip(old.image_id, old.variant))
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    lp = lpips.LPIPS(net="alex", verbose=False).to(dev).eval()
    dists = piq.DISTS(reduction="none").to(dev).eval()

    def to_t(arrs):
        return torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2).float().div(255.0).to(dev)

    rows = []
    for v in variants:
        ids = [i for i in image_ids if (i, v) not in have and variant_path(v, i).exists()]
        for k in range(0, len(ids), batch):
            chunk = ids[k:k + batch]
            ref = [load_rgb(variant_path("clean", i)) for i in chunk]
            out = [load_rgb(variant_path(v, i)) for i in chunk]
            with torch.no_grad():
                r, o = to_t(ref), to_t(out)
                lp_v = lp(o * 2 - 1, r * 2 - 1).flatten().cpu().numpy()
                di_v = dists(o, r).flatten().cpu().numpy()
            for j, iid in enumerate(chunk):
                rows.append({
                    "image_id": iid, "variant": v,
                    "psnr": peak_signal_noise_ratio(ref[j], out[j], data_range=255),
                    "ssim": structural_similarity(ref[j], out[j], channel_axis=2, data_range=255),
                    "lpips": float(lp_v[j]), "dists": float(di_v[j]),
                })
        if ids:
            print(f"{v}: {len(ids)} images scored")
            # checkpoint after each variant so a disconnect loses little
            pd.concat([old, pd.DataFrame(rows)], ignore_index=True).to_csv(out_csv, index=False)
    df = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
    df.to_csv(out_csv, index=False)
    return df

In [ ]:
%%writefile /content/aigi_code/detectors.py
"""AI-generated-image detectors from three families, all scored as 'higher = more fake'.

  UnivFD  - CLIP ViT-L/14 features + linear probe   (Ojha et al., CVPR 2023)
  NPR     - low-level up-sampling artifacts         (Tan et al., CVPR 2024)
  RIGID   - training-free, DINOv2 noise sensitivity (He et al., 2024)

Checkpoints for UnivFD and NPR are the authors' own files, shipped inside their
GitHub repositories (cloned on first use). Each detector writes scores/<name>.csv with
columns image_id, variant, score. Resumable per variant.
"""
import subprocess
import sys

import numpy as np
import pandas as pd

from common import DIRS, load_rgb, set_seed, variant_path

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def _device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def _repo(url):
    """Shallow-clone an authors' repository into weights/ once; return its path."""
    dst = DIRS["weights"] / url.rstrip("/").split("/")[-1]
    if not dst.exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(dst)], check=True)
    return dst


def _clean_state_dict(sd):
    for key in ("model", "state_dict", "net"):
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
            sd = sd[key]
    return {k[7:] if k.startswith("module.") else k: v for k, v in sd.items()}


def _batch_tensor(imgs, crop, mean, std):
    """uint8 HxWx3 list -> normalised NCHW tensor; centre crop to `crop` (no resampling)."""
    import torch
    x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float().div(255.0)
    if crop:
        h, w = x.shape[-2:]
        t, l = (h - crop) // 2, (w - crop) // 2
        x = x[:, :, t:t + crop, l:l + crop]
    m = torch.tensor(mean).view(1, 3, 1, 1)
    s = torch.tensor(std).view(1, 3, 1, 1)
    return (x - m) / s


class UnivFD:
    name = "univfd"

    def __init__(self):
        import open_clip
        import torch
        self.dev = _device()
        # QuickGELU matches the original OpenAI CLIP that UnivFD was trained on.
        model, _, _ = open_clip.create_model_and_transforms("ViT-L-14-quickgelu", pretrained="openai")
        self.clip = model.to(self.dev).eval()
        fc_path = _repo("https://github.com/Yuheng-Li/UniversalFakeDetect") / "pretrained_weights" / "fc_weights.pth"
        sd = _clean_state_dict(torch.load(fc_path, map_location="cpu"))
        sd = {k.split(".")[-1]: v for k, v in sd.items()}  # 'fc.weight' -> 'weight'
        self.fc = torch.nn.Linear(sd["weight"].shape[1], 1)
        self.fc.load_state_dict(sd)
        self.fc = self.fc.to(self.dev).eval()

    def __call__(self, imgs):
        import torch
        x = _batch_tensor(imgs, 224, CLIP_MEAN, CLIP_STD).to(self.dev)
        with torch.no_grad():
            f = self.clip.encode_image(x).float()
            return self.fc(f).flatten().cpu().numpy()


class NPR:
    name = "npr"
    REPO = "https://github.com/chuangchuangtan/NPR-DeepfakeDetection"

    def __init__(self):
        import torch
        self.dev = _device()
        code = _repo(self.REPO)
        sys.path.insert(0, str(code))
        from networks.resnet import resnet50  # from the NPR repository
        self.model = resnet50(num_classes=1)
        sd = _clean_state_dict(torch.load(code / "NPR.pth", map_location="cpu"))
        missing, unexpected = self.model.load_state_dict(sd, strict=False)
        if missing:
            raise RuntimeError(f"NPR weights do not match the network: missing {missing[:5]}")
        if unexpected:
            print("NPR: ignoring unexpected keys", unexpected[:5])
        self.model = self.model.to(self.dev).eval()
        sys.path.remove(str(code))

    def __call__(self, imgs):
        import torch
        # NPR's own test protocol (test.py: no_resize=False, no_crop=True; loadSize=256):
        # PIL bilinear resize to 256x256, no crop, ImageNet normalisation.
        from PIL import Image
        imgs = [np.asarray(Image.fromarray(im).resize((256, 256), Image.BILINEAR)) for im in imgs]
        x = _batch_tensor(imgs, None, IMAGENET_MEAN, IMAGENET_STD).to(self.dev)
        with torch.no_grad():
            return self.model(x).flatten().cpu().numpy()


class RIGID:
    """Score = 1 - cosine(f(x), f(x + lambda*noise)), averaged over a few noise draws."""
    name = "rigid"

    def __init__(self, noise=0.05, draws=3, backbone="dinov2_vitl14"):
        import torch
        self.dev = _device()
        code = _repo("https://github.com/facebookresearch/dinov2")
        # Local clone avoids torch.hub's GitHub API calls; weights still download once.
        self.model = torch.hub.load(str(code), backbone, source="local").to(self.dev).eval()
        self.noise, self.draws = noise, draws

    def __call__(self, imgs):
        import torch
        x = _batch_tensor(imgs, 224, IMAGENET_MEAN, IMAGENET_STD).to(self.dev)
        g = torch.Generator(device=self.dev).manual_seed(2026)  # same noise for every batch
        with torch.no_grad():
            f0 = torch.nn.functional.normalize(self.model(x).float(), dim=-1)
            sims = []
            for _ in range(self.draws):
                eps = torch.randn(x.shape, generator=g, device=self.dev)
                f1 = torch.nn.functional.normalize(self.model(x + self.noise * eps).float(), dim=-1)
                sims.append((f0 * f1).sum(-1))
            return (1 - torch.stack(sims).mean(0)).cpu().numpy()


DETECTORS = {"univfd": UnivFD, "npr": NPR, "rigid": RIGID}


def ids_for(variant, image_ids):
    if variant.startswith("fake_"):
        folder = DIRS["fakes"] / variant[5:]
        return sorted(p.stem for p in folder.glob("*.png"))
    return [i for i in image_ids if variant_path(variant, i).exists()]


def score_all(detector_names, variants, image_ids, batch=16):
    """Score every variant (plus 'clean' and fake_* sets) with each detector."""
    set_seed()
    for name in detector_names:
        out_csv = DIRS["scores"] / f"{name}.csv"
        old = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame(columns=["image_id", "variant", "score"])
        done = set(old.variant.unique())
        todo = [v for v in variants if v not in done]
        if not todo:
            print(f"{name}: all variants already scored")
            continue
        det = DETECTORS[name]()
        frames = [old]
        for v in todo:
            ids = ids_for(v, image_ids)
            if not ids:
                print(f"{name}: {v} has no images yet, skipped")
                continue
            scores = []
            for k in range(0, len(ids), batch):
                chunk = ids[k:k + batch]
                scores.append(det([load_rgb(variant_path(v, i)) for i in chunk]))
            frames.append(pd.DataFrame({"image_id": ids, "variant": v, "score": np.concatenate(scores)}))
            pd.concat(frames, ignore_index=True).to_csv(out_csv, index=False)
            print(f"{name}: {v} ({len(ids)} images)")
        del det
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass


def load_scores():
    frames = []
    for p in sorted(DIRS["scores"].glob("*.csv")):
        d = pd.read_csv(p)
        d["detector"] = p.stem
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

In [ ]:
%%writefile /content/aigi_code/analyze.py
"""Statistics, tables and figures for the paper.

Thresholds are fixed on clean CALIBRATION reals only (95th percentile = 5% FPR);
every reported rate is measured on the held-out TEST reals.

Main outputs in results/:
  table_sanity.csv           detector AUC/TPR on fakes at the threshold (Table 1)
  fpr_by_variant.csv         FPR + 95% bootstrap CI for every detector x variant
  flip_points.csv            lowest strength with FPR significantly above clean
  neural_vs_classical.csv    paired McNemar tests at equal degradation, BH-corrected
  fidelity_verdict_gap.csv   FPR among near-identical outputs (LPIPS<0.1, SSIM>0.9)
  fpr_by_lpips_bin.csv       neural vs classical inside LPIPS bins (matched fidelity)
  logit_neural_effect.csv    odds ratio of 'neural' holding LPIPS fixed
  fig_dose.(png|pdf), fig_lpips.(png|pdf), fig_examples.(png|pdf)
"""
import numpy as np
import pandas as pd
from scipy.stats import binomtest, mannwhitneyu
from statsmodels.stats.multitest import multipletests

from common import DIRS, get_splits, load_rgb, parse_variant, variant_path

RNG = np.random.default_rng(2026)
N_BOOT = 2000
ALPHA = 0.05

# Fixed colour per method (categorical slots, never cycled) + marker for print.
METHOD_STYLE = {
    "bicubic": ("#2a78d6", "o"), "nlm": ("#2a78d6", "o"),
    "lanczosusm": ("#eb6834", "s"), "bm3d": ("#eb6834", "s"),
    "swinir": ("#1baf7a", "^"),
    "realesrgan": ("#6250d6", "D"), "restormer": ("#6250d6", "D"),
    "sdx4": ("#e87ba4", "v"),
    "noisy": ("#8a8a8a", "x"),
}
METHOD_LABEL = {"bicubic": "Bicubic", "lanczosusm": "Lanczos+USM", "realesrgan": "Real-ESRGAN",
                "swinir": "SwinIR", "sdx4": "SD x4 upscaler", "noisy": "Noisy input",
                "nlm": "NLM", "bm3d": "BM3D", "restormer": "Restormer"}
DET_LABEL = {"univfd": "UnivFD", "npr": "NPR", "rigid": "RIGID"}
FAMILY_LABEL = {"up": "Upscaling (factor)", "dn": "Denoising (noise sigma)"}


# ----------------------------------------------------------------- helpers
def boot_ci(flags):
    flags = np.asarray(flags, dtype=float)
    n = len(flags)
    if n == 0:
        return np.nan, np.nan
    idx = RNG.integers(0, n, size=(N_BOOT, n))
    m = flags[idx].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def mcnemar(a, b):
    """Exact McNemar on paired booleans. Returns (b_only_a, b_only_b, p)."""
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return n01, n10, p


def thresholds(scores, calib_ids, mode="fpr5"):
    """Per-detector threshold. 'fpr5': 95th pct of clean calibration reals.
    'eer': equal-error point between clean calibration reals and all fakes (robustness check)."""
    out = {}
    for det, d in scores.groupby("detector"):
        real = d[(d.variant == "clean") & d.image_id.isin(calib_ids)].score.values
        if mode == "fpr5":
            out[det] = float(np.quantile(real, 0.95))
        else:
            fake = d[d.variant.str.startswith("fake_")].score.values
            if len(fake) == 0:
                raise ValueError("EER mode needs scored fakes")
            cands = np.unique(np.concatenate([real, fake]))
            fpr = np.array([(real > t).mean() for t in cands])
            fnr = np.array([(fake <= t).mean() for t in cands])
            out[det] = float(cands[np.argmin(np.abs(fpr - fnr))])
    return out


def build_table(scores, fidelity, splits, mode="fpr5"):
    """One row per detector x variant x image, with flag, metadata and fidelity."""
    tau = thresholds(scores, splits["calib"], mode)
    d = scores.copy()
    d["flag"] = d.score > d.detector.map(tau)
    meta = pd.DataFrame([dict(variant=v, **parse_variant(v)) for v in d.variant.unique()])
    d = d.merge(meta, on="variant", how="left")
    if fidelity is not None and len(fidelity):
        d = d.merge(fidelity, on=["image_id", "variant"], how="left")
    if "lpips" in d:
        d.loc[d.variant == "clean", ["lpips", "dists"]] = 0.0
        d.loc[d.variant == "clean", ["ssim"]] = 1.0
    test, calib = set(splits["test"]), set(splits["calib"])
    d["split"] = np.select(
        [d.variant.str.startswith("fake_"), d.image_id.isin(test), d.image_id.isin(calib)],
        ["fake", "test", "calib"], default="unused")
    return d, tau


# ------------------------------------------------------------------ tables
def sanity_table(d):
    rows = []
    for det, g in d.groupby("detector"):
        real = g[(g.variant == "clean") & (g.split == "test")]
        fakes = g[g.family == "fake"]
        row = {"detector": det, "clean_test_fpr": real.flag.mean(), "n_real": len(real)}
        if len(fakes):
            u = mannwhitneyu(fakes.score, real.score, alternative="two-sided").statistic
            row.update(auc=u / (len(fakes) * len(real)), tpr_at_tau=fakes.flag.mean(), n_fake=len(fakes))
            for gen, gg in fakes.groupby("method"):
                row[f"tpr_{gen}"] = gg.flag.mean()
        rows.append(row)
    return pd.DataFrame(rows)


def fpr_table(d):
    t = d[d.split == "test"]
    rows = []
    for (det, v), g in t.groupby(["detector", "variant"]):
        lo, hi = boot_ci(g.flag)
        m = parse_variant(v)
        rows.append({"detector": det, "variant": v, **m, "n": len(g), "fpr": g.flag.mean(),
                     "ci_lo": lo, "ci_hi": hi, "mean_score": g.score.mean(),
                     "median_lpips": g.lpips.median() if "lpips" in g else np.nan})
    return pd.DataFrame(rows).sort_values(["detector", "family", "method", "strength"])


def flip_points(d):
    """Lowest strength at which FPR rises significantly above the same images' clean FPR."""
    t = d[d.split == "test"]
    clean = t[t.variant == "clean"].set_index(["detector", "image_id"]).flag
    tests = []
    for (det, fam, meth, s), g in t[t.kind.isin(["neural", "classical", "baseline"]) & (t.family != "clean")] \
            .groupby(["detector", "family", "method", "strength"]):
        c = clean.loc[det].reindex(g.image_id).values
        n01, n10, p = mcnemar(g.flag.values, c)
        tests.append({"detector": det, "family": fam, "method": meth, "strength": s,
                      "fpr": g.flag.mean(), "clean_fpr": np.mean(c), "newly_flagged": n01,
                      "newly_cleared": n10, "p": p})
    tests = pd.DataFrame(tests)
    if tests.empty:
        return tests, tests
    tests["q"] = multipletests(tests.p, method="fdr_bh")[1]
    tests["sig_increase"] = (tests.q < ALPHA) & (tests.fpr > tests.clean_fpr)
    flips = (tests[tests.sig_increase].groupby(["detector", "family", "method"]).strength.min()
             .rename("flip_strength").reset_index())
    return tests, flips


def neural_vs_classical(d):
    t = d[d.split == "test"]
    rows = []
    for (det, fam, s), g in t[t.kind.isin(["neural", "classical"])].groupby(["detector", "family", "strength"]):
        piv = g.pivot_table(index="image_id", columns="method", values="flag", aggfunc="first")
        kinds = g.drop_duplicates("method").set_index("method").kind
        for nm in [m for m in piv.columns if kinds[m] == "neural"]:
            for cm in [m for m in piv.columns if kinds[m] == "classical"]:
                pair = piv[[nm, cm]].dropna()
                if pair.empty:
                    continue
                n01, n10, p = mcnemar(pair[nm].astype(bool), pair[cm].astype(bool))
                rows.append({"detector": det, "family": fam, "strength": s, "neural": nm, "classical": cm,
                             "n": len(pair), "fpr_neural": pair[nm].mean(), "fpr_classical": pair[cm].mean(),
                             "diff": pair[nm].mean() - pair[cm].mean(),
                             "only_neural_flags": n01, "only_classical_flags": n10, "p": p})
    r = pd.DataFrame(rows)
    if len(r):
        r["q"] = multipletests(r.p, method="fdr_bh")[1]
    return r


def fidelity_verdict_gap(d, lpips_max=0.10, ssim_min=0.90):
    t = d[(d.split == "test") & d.kind.isin(["neural", "classical"]) & (d.lpips < lpips_max) & (d.ssim > ssim_min)]
    rows = []
    for (det, kind), g in t.groupby(["detector", "kind"]):
        lo, hi = boot_ci(g.flag)
        rows.append({"detector": det, "kind": kind, "n": len(g), "fpr": g.flag.mean(), "ci_lo": lo, "ci_hi": hi})
    return pd.DataFrame(rows)


def lpips_bins(d, n_bins=5):
    """Matched fidelity: bin restored outputs by LPIPS (pooled edges per family),
    then compare neural vs classical FPR inside each bin."""
    t = d[(d.split == "test") & d.kind.isin(["neural", "classical"])].copy()
    rows = []
    for fam, g in t.groupby("family"):
        edges = np.unique(np.quantile(g.lpips.dropna(), np.linspace(0, 1, n_bins + 1)))
        g = g.assign(bin=pd.cut(g.lpips, edges, include_lowest=True, labels=False))
        for (det, b, kind), gg in g.groupby(["detector", "bin", "kind"]):
            lo, hi = boot_ci(gg.flag)
            rows.append({"detector": det, "family": fam, "bin": int(b), "kind": kind,
                         "lpips_lo": edges[int(b)], "lpips_hi": edges[int(b) + 1],
                         "lpips_median": gg.lpips.median(), "n": len(gg),
                         "fpr": gg.flag.mean(), "ci_lo": lo, "ci_hi": hi})
    return pd.DataFrame(rows)


def lpips_overlap(d):
    """How much neural and classical outputs overlap in LPIPS, per family.
    Matched-fidelity claims are only made inside the common support."""
    t = d[(d.split == "test") & d.kind.isin(["neural", "classical"]) & (d.detector == d.detector.iloc[0])]
    rows = []
    for fam, g in t.groupby("family"):
        q = {k: gk.lpips.dropna() for k, gk in g.groupby("kind")}
        if len(q) < 2:
            continue
        lo = max(q["neural"].min(), q["classical"].min())
        hi = min(q["neural"].max(), q["classical"].max())
        for kind, v in q.items():
            rows.append({"family": fam, "kind": kind, "n": len(v), "min": v.min(), "q25": v.quantile(.25),
                         "median": v.median(), "q75": v.quantile(.75), "max": v.max(),
                         "common_lo": lo, "common_hi": hi,
                         "share_in_common_support": float(((v >= lo) & (v <= hi)).mean()) if hi > lo else 0.0})
    return pd.DataFrame(rows)


def lpips_bin_tests(bins, min_n=20):
    """Neural vs classical inside each LPIPS bin (Fisher exact, BH-corrected).
    Bins where either kind has fewer than min_n outputs are marked and not tested."""
    from scipy.stats import fisher_exact
    rows = []
    for (det, fam, b), g in bins.groupby(["detector", "family", "bin"]):
        k = g.set_index("kind")
        if not {"neural", "classical"} <= set(k.index):
            continue
        nn, nc = int(k.loc["neural", "n"]), int(k.loc["classical", "n"])
        fn, fc = k.loc["neural", "fpr"], k.loc["classical", "fpr"]
        ok = nn >= min_n and nc >= min_n
        p = np.nan
        if ok:
            an, ac = round(fn * nn), round(fc * nc)
            p = fisher_exact([[an, nn - an], [ac, nc - ac]])[1]
        rows.append({"detector": det, "family": fam, "bin": b, "lpips_lo": k.lpips_lo.iloc[0],
                     "lpips_hi": k.lpips_hi.iloc[0], "n_neural": nn, "n_classical": nc,
                     "fpr_neural": fn, "fpr_classical": fc, "diff": fn - fc, "overlap_ok": ok, "p": p})
    r = pd.DataFrame(rows)
    if len(r) and r.overlap_ok.any():
        m = r.overlap_ok
        r.loc[m, "q"] = multipletests(r.loc[m, "p"], method="fdr_bh")[1]
    return r


def _fit(formula, g):
    import statsmodels.formula.api as smf
    return smf.logit(formula, data=g).fit(disp=0, cov_type="cluster",
                                          cov_kwds={"groups": pd.factorize(g.image_id)[0]})


def _or(fit, term):
    ci = fit.conf_int().loc[term]
    return np.exp(fit.params[term]), np.exp(ci[0]), np.exp(ci[1]), fit.pvalues[term]


MODELS = {
    # primary (Eq. 2): neural indicator, standardised LPIPS, degradation condition (family x strength)
    "primary": "y ~ neural + lpips_z + C(cond)",
    # sensitivity: family only, no strength adjustment (the earlier specification)
    "family_only": "y ~ neural + lpips_z + C(family)",
}


def neural_effect(d, metric="lpips"):
    """Logistic regressions per detector with standard errors clustered by image.

    primary      : flag ~ neural + LPIPS* + condition       (condition = family x strength, 4 levels)
    family_only  : flag ~ neural + LPIPS* + family           (sensitivity, no strength adjustment)
    interaction  : flag ~ neural * family + LPIPS* + condition -> neural OR separately per family
    common       : primary model on outputs inside the LPIPS range both kinds cover, per condition
    """
    t = d[(d.split == "test") & d.kind.isin(["neural", "classical"])].dropna(subset=[metric]).copy()
    t["neural"] = (t.kind == "neural").astype(int)
    # the matching variable (LPIPS by default); the column keeps its name so the formulas stay the same
    t["lpips_z"] = (t[metric] - t[metric].mean()) / t[metric].std()
    t["match"] = t[metric]
    t["y"] = t.flag.astype(int)
    t["cond"] = t.family + "_" + t.strength.astype(str)
    rows = []
    for det, g in t.groupby("detector"):
        row = {"detector": det, "matching_metric": metric, "n": len(g), "n_images": g.image_id.nunique()}
        for name, f in MODELS.items():
            try:
                o, lo, hi, p = _or(_fit(f, g), "neural")
                row.update({f"or_{name}": o, f"or_{name}_lo": lo, f"or_{name}_hi": hi, f"p_{name}": p})
            except Exception as e:  # e.g. perfect separation when a detector never fires
                row[f"err_{name}"] = repr(e)
        try:
            fi = _fit("y ~ neural * C(family) + lpips_z + C(cond)", g)
            fams = sorted(g.family.unique())
            b, V = fi.params, fi.cov_params()
            for fam in fams:
                inter = f"neural:C(family)[T.{fam}]"
                est = b["neural"] + (b[inter] if inter in b else 0.0)
                var = V.loc["neural", "neural"] + ((V.loc[inter, inter] + 2 * V.loc["neural", inter]) if inter in b else 0.0)
                se = np.sqrt(var)
                row.update({f"or_neural_{fam}": np.exp(est), f"or_neural_{fam}_lo": np.exp(est - 1.96 * se),
                            f"or_neural_{fam}_hi": np.exp(est + 1.96 * se)})
            inter_terms = [k for k in b.index if k.startswith("neural:")]
            row["p_interaction"] = fi.pvalues[inter_terms[0]] if inter_terms else np.nan
        except Exception as e:
            row["err_interaction"] = repr(e)
        # common LPIPS support within each degradation condition
        keep = []
        for _, gc in g.groupby("cond"):
            lo = max(gc[gc.neural == 1].match.min(), gc[gc.neural == 0].match.min())
            hi = min(gc[gc.neural == 1].match.max(), gc[gc.neural == 0].match.max())
            keep.append(gc[(gc.match >= lo) & (gc.match <= hi)])
        gcs = pd.concat(keep)
        row["n_common"] = len(gcs)
        try:
            o, lo, hi, p = _or(_fit(MODELS["primary"], gcs), "neural")
            row.update(or_common=o, or_common_lo=lo, or_common_hi=hi, p_common=p)
        except Exception as e:
            row["err_common"] = repr(e)
        rows.append(row)
    return pd.DataFrame(rows)


# ----------------------------------------------------------------- figures
def _style():
    import matplotlib as mpl
    mpl.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#6b6b6b",
        "axes.linewidth": 0.6, "xtick.color": "#4a4a4a", "ytick.color": "#4a4a4a",
        "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.5,
        "legend.frameon": False, "savefig.dpi": 300, "savefig.bbox": "tight",
    })


def _save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(DIRS["results"] / f"{name}.{ext}")


def fig_dose(fpr):
    """FPR vs strength: rows = enhancement family, columns = detector.
    Neural = solid line + filled marker; classical = dashed + hollow; grey band = clean-image FPR CI."""
    import matplotlib.pyplot as plt
    _style()
    dets = sorted(fpr.detector.unique())
    fams = [f for f in ("up", "dn") if f in set(fpr.family)]
    fig, axes = plt.subplots(len(fams), len(dets), figsize=(2.3 * len(dets), 2.1 * len(fams)),
                             squeeze=False, sharey=True)
    for i, fam in enumerate(fams):
        for j, det in enumerate(dets):
            ax = axes[i][j]
            clean = fpr[(fpr.detector == det) & (fpr.variant == "clean")]
            if len(clean):
                ax.axhspan(clean.ci_lo.iloc[0], clean.ci_hi.iloc[0], color="#d9d9d9", lw=0, zorder=0)
                ax.axhline(clean.fpr.iloc[0], color="#8a8a8a", lw=0.8, zorder=1)
            g = fpr[(fpr.detector == det) & (fpr.family == fam)]
            for meth, gm in g.groupby("method"):
                gm = gm.sort_values("strength")
                col, mk = METHOD_STYLE.get(meth, ("#4a4a4a", "o"))
                kind = gm.kind.iloc[0]
                ls = "-" if kind == "neural" else ("--" if kind == "classical" else ":")
                face = col if kind == "neural" else "white"
                ax.fill_between(gm.strength, gm.ci_lo, gm.ci_hi, color=col, alpha=0.12, lw=0)
                ax.plot(gm.strength, gm.fpr, ls=ls, color=col, lw=1.5, marker=mk, ms=4.5,
                        mfc=face, mec=col, mew=1.2, label=METHOD_LABEL.get(meth, meth))
            xs = sorted(g.strength.unique())
            ax.set_xticks(xs)
            ax.set_ylim(0, 1)
            if i == 0:
                ax.set_title(DET_LABEL.get(det, det))
            if j == 0:
                ax.set_ylabel(f"False-positive rate\n{FAMILY_LABEL[fam].split(' (')[0]}")
            ax.set_xlabel(FAMILY_LABEL[fam].split("(")[1].rstrip(")"))
        handles, labels = axes[i][-1].get_legend_handles_labels()
        axes[i][-1].legend(handles, labels, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=7)
    fig.tight_layout()
    _save(fig, "fig_dose")
    return fig


def fig_lpips(bins):
    import matplotlib.pyplot as plt
    _style()
    dets = sorted(bins.detector.unique())
    fams = [f for f in ("up", "dn") if f in set(bins.family)]
    fig, axes = plt.subplots(len(fams), len(dets), figsize=(2.3 * len(dets), 2.1 * len(fams)),
                             squeeze=False, sharey=True)
    styles = {"neural": ("#6250d6", "-", "D", "#6250d6"), "classical": ("#2a78d6", "--", "o", "white")}
    for i, fam in enumerate(fams):
        for j, det in enumerate(dets):
            ax = axes[i][j]
            g = bins[(bins.detector == det) & (bins.family == fam)]
            for kind, gk in g.groupby("kind"):
                col, ls, mk, face = styles[kind]
                gk = gk.sort_values("lpips_median")
                ax.errorbar(gk.lpips_median, gk.fpr, yerr=[gk.fpr - gk.ci_lo, gk.ci_hi - gk.fpr],
                            color=col, ls=ls, lw=1.5, marker=mk, ms=4.5, mfc=face, mec=col,
                            capsize=2, elinewidth=0.8, label=kind.capitalize())
            ax.set_ylim(0, 1)
            if i == 0:
                ax.set_title(DET_LABEL.get(det, det))
            if j == 0:
                ax.set_ylabel(f"False-positive rate\n{FAMILY_LABEL[fam].split(' (')[0]}")
            ax.set_xlabel("LPIPS to original (bin median)")
        axes[i][-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=7)
    fig.tight_layout()
    _save(fig, "fig_lpips")
    return fig


def pick_example(d, variant="up_realesrgan_4"):
    """Systematic example choice (no cherry-picking): the test image whose LPIPS for
    `variant` is closest to that variant's median LPIPS. Ties broken by image id."""
    t = d[(d.split == "test") & (d.variant == variant)].drop_duplicates("image_id")
    med = t.lpips.median()
    t = t.assign(dist=(t.lpips - med).abs()).sort_values(["dist", "image_id"])
    return t.image_id.iloc[0]


def leakage_checks(splits, scores):
    """Assertions that calibration and test never mix."""
    calib, test = set(splits["calib"]), set(splits["test"])
    checks = {
        "calib_test_disjoint": len(calib & test) == 0,
        "n_calib": len(calib), "n_test": len(test),
        "no_enhanced_calibration_images": not bool(
            scores[(scores.variant != "clean") & ~scores.variant.str.startswith("fake_")]
            .image_id.isin(calib).any()),
        "fakes_disjoint_from_reals": not scores[scores.variant.str.startswith("fake_")].image_id.isin(calib | test).any(),
    }
    return checks


def fig_examples(d, image_id, variants=("clean", "up_bicubic_4", "up_lanczosusm_4", "up_swinir_4", "up_realesrgan_4")):
    """One test image through several operators, with LPIPS and each detector's verdict."""
    import matplotlib.pyplot as plt
    _style()
    variants = [v for v in variants if variant_path(v, image_id).exists()]
    fig, axes = plt.subplots(1, len(variants), figsize=(2.0 * len(variants), 2.6))
    for ax, v in zip(np.atleast_1d(axes), variants):
        img = load_rgb(variant_path(v, image_id))
        ax.imshow(img[128:384, 128:384])  # central 256 px crop so texture is visible
        ax.set_xticks([]), ax.set_yticks([]), ax.grid(False)
        r = d[(d.image_id == image_id) & (d.variant == v)]
        verdict = "  ".join(f"{DET_LABEL.get(x.detector, x.detector)}:{'FAKE' if x.flag else 'real'}"
                            for x in r.sort_values("detector").itertuples())
        lp = r.lpips.iloc[0] if len(r) and "lpips" in r.columns else np.nan
        meta = parse_variant(v)
        unit = "x" if meta["family"] == "up" else "sigma "
        name = "Original" if v == "clean" else f"{METHOD_LABEL.get(meta['method'])} ({unit}{meta['strength']})"
        ax.set_title(f"{name}\nLPIPS {'n/a' if pd.isna(lp) else f'{lp:.3f}'}", fontsize=8)
        ax.set_xlabel(verdict, fontsize=6)
    fig.tight_layout()
    _save(fig, "fig_examples")
    return fig


# ------------------------------------------------------------------- driver
def run_all(mode="fpr5", make_figures=True):
    from detectors import load_scores
    splits = get_splits()
    scores = load_scores()
    fid_path = DIRS["results"] / "fidelity.csv"
    fidelity = pd.read_csv(fid_path) if fid_path.exists() else None
    d, tau = build_table(scores, fidelity, splits, mode)
    suffix = "" if mode == "fpr5" else f"_{mode}"
    out = {
        "thresholds": pd.DataFrame({
            "detector": list(tau), "tau": list(tau.values()),
            "rule": "95th pct of clean calib (linear interp.), flag if score > tau" if mode == "fpr5"
                    else "EER between clean calib and Synthbuster (no test images)",
            "calib_fpr_observed": [d[(d.detector == k) & (d.split == "calib") & (d.variant == "clean")].flag.mean() for k in tau],
            "scores_equal_to_tau": [int((scores[scores.detector == k].score == tau[k]).sum()) for k in tau]}),
        "table_sanity": sanity_table(d),
        "fpr_by_variant": fpr_table(d),
    }
    tests, flips = flip_points(d)
    out["flip_tests"], out["flip_points"] = tests, flips
    out["neural_vs_classical"] = neural_vs_classical(d)
    if fidelity is not None:
        out["fidelity_verdict_gap"] = fidelity_verdict_gap(d)
        out["fpr_by_lpips_bin"] = lpips_bins(d)
        out["lpips_bin_tests"] = lpips_bin_tests(out["fpr_by_lpips_bin"])
        out["lpips_overlap"] = lpips_overlap(d)
        out["logit_neural_effect"] = neural_effect(d)
        # robustness: match on DISTS or PSNR instead of LPIPS (metrics disagree for upscaling)
        out["logit_neural_effect_by_metric"] = pd.concat(
            [neural_effect(d, m) for m in ("lpips", "dists", "psnr", "ssim")], ignore_index=True)
    checks = leakage_checks(splits, scores)
    out["leakage_checks"] = pd.DataFrame([checks])
    # per-image scores and binary flags, for replication
    keep = [c for c in ["detector", "image_id", "variant", "split", "family", "method", "strength", "kind",
                        "score", "flag", "psnr", "ssim", "lpips", "dists"] if c in d.columns]
    out["per_image"] = d[keep]
    for k, v in out.items():
        v.to_csv(DIRS["results"] / f"{k}{suffix}.csv", index=False)
    if make_figures and mode == "fpr5":
        fig_dose(out["fpr_by_variant"])
        if fidelity is not None:
            fig_lpips(out["fpr_by_lpips_bin"])
    print("Leakage checks:", checks)
    print("Thresholds:", {k: round(v, 4) for k, v in tau.items()})
    print("\nSanity check (fakes):\n", out["table_sanity"].round(3).to_string(index=False))
    return d, out

In [ ]:
import sys, importlib
sys.path.insert(0, CODE)
import common, data, enhance, fidelity, detectors, analyze
for m in (common, data, enhance, fidelity, detectors, analyze):
    importlib.reload(m)
common.ensure_dirs()
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4')

## Step 1 — Check data already on Drive

In [ ]:
splits = common.get_splits()
print(len(common.list_clean_ids()), 'clean photos |', len(splits['calib']), 'calibration |', len(splits['test']), 'test (small study)')
print('fake generators:', data.fake_generators())
assert len(splits['test']) == 300, 'Expected 300 test images - send this output to Claude'

In [ ]:
# Exclusions: which RAISE images did not make it into the clean set (for the paper's Data section)
import csv, json
from pathlib import Path
raise_csv = common.ROOT / 'RAISE_1k.csv'
exclusions = {}
if raise_csv.exists():
    with open(raise_csv, newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    names = {data._safe_id('r', Path((r.get('File') or '').strip()).stem) for r in rows if (r.get('File') or '').strip()}
    have = set(common.list_clean_ids())
    missing = sorted(names - have)
    exclusions = {'raise_rows': len(rows), 'usable': len(have & names), 'excluded': len(missing), 'excluded_ids': missing}
    print(f"RAISE rows: {len(rows)} | usable: {len(have & names)} | excluded: {len(missing)}")
(common.ROOT / 'exclusions.json').write_text(json.dumps(exclusions, indent=1))

## Reproducibility record
Software versions, GPU, seeds and checkpoint hashes, saved to `results/environment.json` for the paper.

In [ ]:
import json, hashlib, platform, importlib.metadata as md_
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]
pkgs = ['torch', 'torchvision', 'open_clip_torch', 'spandrel', 'spandrel_extra_arches', 'lpips', 'piq',
        'scikit-image', 'opencv-python-headless', 'numpy', 'scipy', 'statsmodels', 'pandas', 'Pillow']
env = {'python': platform.python_version(),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'seed': common.SEED, 'crop': common.CROP, 'resize_short': common.RESIZE_SHORT,
       'n_calib': len(splits['calib']), 'n_test': len(splits['test']),
       'packages': {p: md_.version(p) for p in pkgs if p in {d.metadata['Name'] for d in md_.distributions()}},
       'weights': {}}
for p in sorted(common.DIRS['weights'].rglob('*.pth')):
    env['weights'][str(p.relative_to(common.DIRS['weights']))] = sha(p)
(common.DIRS['results'] / 'environment.json').write_text(json.dumps(env, indent=1))
print(json.dumps(env, indent=1)[:1500])

## Step 2 — Enhancement variants
Prints seconds/image after the first 5 images of each variant. Skips anything already made.

In [ ]:
nlm_h = enhance.tune_nlm(splits['calib'])
failed = enhance.run_variants(enhance.SMALL_VARIANTS, splits['test'], nlm_h)
print('ALL VARIANTS DONE' if not failed else 'Send the failed lines to Claude')

## Step 3 — Fidelity

In [ ]:
variants = enhance.SMALL_VARIANTS
fid = fidelity.compute_fidelity(variants, splits['test'])
fid[fid.variant.isin(variants)].groupby('variant')[['psnr', 'ssim', 'lpips', 'dists']].median().round(3)

## Step 4 — Detector scores
`clean` = all real photos (calibration + test); `fake_*` = Synthbuster sanity sets.

In [ ]:
to_score = ['clean'] + variants + [f'fake_{g}' for g in data.fake_generators()]
detectors.score_all(['univfd', 'npr', 'rigid'], to_score, common.list_clean_ids(), batch=16)

## Step 5 — Analysis
Results go to `MyDrive/aigi_study/results/`. Send Claude the downloaded `results.zip`.

In [ ]:
# keep only the small-study variants in the analysis
import pandas as pd
_load = detectors.load_scores
def load_small():
    s = _load()
    return s[s.variant.isin(['clean'] + variants) | s.variant.str.startswith('fake_')]
detectors.load_scores = load_small

d, out = analyze.run_all(mode='fpr5')
_ = analyze.run_all(mode='eer', make_figures=False)
from IPython.display import Image, display
display(Image(str(common.DIRS['results'] / 'fig_dose.png')))
display(Image(str(common.DIRS['results'] / 'fig_lpips.png')))

In [ ]:
pd.set_option('display.width', 200)
print('Neural vs classical (paired McNemar, BH-corrected):'); display(out['neural_vs_classical'].round(3))
print('Neural effect holding LPIPS fixed (odds ratio):');    display(out['logit_neural_effect'].round(3))
print('Near-identical outputs still flagged:');               display(out['fidelity_verdict_gap'].round(3))
print('LPIPS overlap between neural and classical outputs:'); display(out['lpips_overlap'].round(3))
print('Neural vs classical inside LPIPS bins (Fisher, BH):'); display(out['lpips_bin_tests'].round(3))
print('Leakage checks:', out['leakage_checks'].to_dict('records')[0])

In [ ]:
# Fig. 4: chosen by a fixed rule, not by eye: the test image whose Real-ESRGAN x4 LPIPS is closest to the median
example_id = analyze.pick_example(d, 'up_realesrgan_4')
print('example image:', example_id)
analyze.fig_examples(d, example_id)

In [ ]:
!cd "$STUDY_ROOT" && zip -qr /content/results.zip results splits.json nlm_h_v2.json exclusions.json
from google.colab import files; files.download('/content/results.zip')